In [ ]:
import os
import sys
sys.path.append('/datasets/')

import math
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from classifier import MLP
from dataset.MNIST import MNIST
from dataset.FMNIST import FMNIST
from dataset.CIFAR10 import CIFAR10
# from torchvision.datasets import MNIST
from dataset.MNISTPerClass import MNISTPerClass
from dataset.FMNISTPerClass import FMNISTPerClass
from dataset.CIFAR10PerClass import CIFARPerClass
from Autoencoder_functions import koopman_loss, collect_latent_states
from torch.nn.utils import parameters_to_vector
from scipy.linalg import eig, inv
from torch.utils.tensorboard import SummaryWriter
from torch.nn.utils.stateless import functional_call
from tensorboard import notebook

from tqdm import tqdm



In [ ]:
model = 0 # 0: nominal | 1: deeper
if model == 0:
    from Autoencoder_real import KoopmanAutoencoder
elif model == 1:
    from Autoencoder_real_v2 import KoopmanAutoencoder

# Use GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cuda:1')
print('Currently using... '+str(device))

Info = [3, -1, -1, 3, 11, -1, 'CIFAR10'] # kae_coef, sub_coef, eig_coef, kae_classifier_coef, hidden_k, num_mode_dom, dataset (MNIST, FMNIST, CIFAR10)


Initializations

In [ ]:
def classifier_sub(x, p_vec):
    idx, p_recon = 0, []
    for layer in classifier_shapes:
        layer_params = []
        for shape in layer:
            offset = np.prod(shape)
            layer_params.append(p_vec[idx:idx+offset].reshape(shape))
            idx += offset
        p_recon.append(layer_params)
    
    w0, b0 = p_recon[0]
    w1, b1 = p_recon[1]
    x = F.linear(x, w0, b0)
    x = F.relu(x)
    x = F.linear(x, w1, b1)
    return x

def compute_l_kae(kae, params_snapshots):
    x = torch.stack(params_snapshots, dim=0).to(device)
    latents, latents_next = collect_latent_states(kae, x)
    kae.compute_koopman_operator(latents, latents_next)
    x_hat, z, z_pred = kae(x)
    recon_loss, state_pred_loss, koopman_pred_loss = koopman_loss(x, x_hat, z_pred, p, kae)
    loss_kae = c1*recon_loss + c2*state_pred_loss + c3*koopman_pred_loss # + c4*k_norm_loss      
    return loss_kae, z


def compute_theta_sub_all(kae, z, ko):
    eigvals, eigvec_left = torch.linalg.eig(ko)
    eigvec_left = eigvec_left.real.detach()
    # B = np.pad(np.eye(n_params), ((0, 0), (0, N_O - n_params)), mode='constant')
    eigvec_left_inv = torch.linalg.pinv(eigvec_left)
    v = (kae.decoder(eigvec_left_inv)).T
    phi = eigvec_left @ z[-1, :]
    param_sub_all = v @ torch.diag(phi)
    return param_sub_all, eigvals

def test_classifier(model, test_loader):
    model.eval()  # evaluation mode
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f'Test Accuracy: {accuracy:.2f}%')


if __name__=='__main__':
    # Set training to be deterministic
    seed = 10
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    # notebook.start("--logdir runs")
    writer = SummaryWriter(log_dir='results/log_sub')

    # Fixed parameters (do not change)
    batch_size = 128 # 128
    lr_classifier = 1e-3 # 1e-5
    T = 5 # 20 
    p = T # 20
    # max_param_stack = 2**9 #batch_size - p - 1
    max_param_threshold = 0.15
    kae_break_threshold = 0.005
    c1, c2, c3 = 1, 1, 1
    image_size = 784  # 28x28 images flattened
    # hidden_c = 16
    num_classes = 10
    target_dataset = Info[-1]

    # Info = [kae_coef, sub_coef, eig_coef, kae_classifier_coef, hidden_k, num_mode_dom]
    # for i in range(len(Info)):
    #     Info[i] = int(Info[i])

    # Varialbe initialization (do not change)
    stack_param = True
    learn_kae = False
    save = True
    
    # Load datasets
    if target_dataset == 'MNIST':
        dataset_in_use_per_class = MNISTPerClass(batch_size=batch_size)
        dataset_in_use = MNIST(batch_size=batch_size)
        max_param_stack = 2 ** 9
        hidden_c = 16

    elif target_dataset == 'FMNIST':
        dataset_in_use_per_class = FMNISTPerClass(batch_size=batch_size)
        dataset_in_use = FMNIST(batch_size=batch_size)
        max_param_stack = 2 ** 9
        hidden_c = 16

    elif target_dataset == 'CIFAR10':
        dataset_in_use_per_class = CIFARPerClass(batch_size=batch_size)
        dataset_in_use = CIFAR10(batch_size=batch_size)
        max_param_stack = 2 ** 9
        hidden_c = 16

    # Define KAE test classifier
    kae_classifier = MLP(image_size, hidden_c, num_classes).to(device)
    kae_classifier.eval()

    # Build the classifier
    classifier = torch.load('results/classifier_'+str(target_dataset)+'_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
    stack_param = False
    learn_kae = True
    print('Load original classifier...')
    criterion_classifier = nn.CrossEntropyLoss()
    optimizer_classifier = optim.Adam(classifier.parameters(), lr=lr_classifier)
    classifier_shapes = [
        [(hidden_c, image_size), (hidden_c)],
        [(num_classes, hidden_c), (num_classes)]
    ]
    param_vec = parameters_to_vector(classifier.parameters()) 
    state_dim = param_vec.shape[0]    

    # Load parameter history
    train_loader_classifier = dataset_in_use.train_loader
    test_loader = dataset_in_use.test_loader
    temp=torch.load('results/params_snapshots_'+str(target_dataset)+'_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
    params_snapshots = temp['params_snapshots']
    test_classifier(classifier, test_loader)
    n_params = len(params_snapshots[0])
    print(n_params)
    classifier.train()
    
    n_batch = len(train_loader_classifier)


Load trained KAE

In [ ]:
# LOAD KAE
# [kae_coef, sub_coef, eig_coef, kae_classifier_coef, hidden_k, num_mode_dom]
print(Info)
Info = Info[:-1]
hidden_k = Info[4]
kae = KoopmanAutoencoder(state_dim=state_dim, hidden_dim=hidden_k).to(device)

# kae = torch.load('results/[Candidate]kae_[10, 0, 0, 100, 64, 2].pth', weights_only=False, map_location=device)

# kae = torch.load('results/kae_'+str(Info)+'_deepr.pth', weights_only=False, map_location=device)
# kae = torch.load('results/kae_'+str(Info)+'_continue.pth', weights_only=False, map_location=device)
# kae = torch.load('waypoints/kae_v2_'+str(Info)+'_continue.pth', weights_only=False, map_location=device)
kae = torch.load('results/kae_'+str(target_dataset)+'_'+str(Info)+'.pth', weights_only=False, map_location=device)

# 500 Iterations, original: 94.27%, 512 parameter history
# wo sub 
# [O][10, 0, 0, 100, 64, 2]_cont: 85.9%, overlap 0 / decomp lowest 100%
# [O][10, 0, 0, 100, 32, 2]: 88.87%, overlap 0 / decomp lowest 91.97% 
# [O][10, 0, 0, 100, 128, 2]: 92%, overlap 0 / decomp lowest 99% - 2650 ver

# [ ][10, 0, 0, 1, 16, 2]
# [ ][10, 0, 0, 1, 32, 2]
# [ ][10, 0, 0, 1, 64, 2]
# [ ][100, 0, 0, 1, 64, 2]
# [ ][10, 0, 0, 1, 128, 2]
# [ ][100, 0, 0, 1, 128, 2]

# wo sub deepr
# [?][10, 0, 0, 100, 10, 2]
# [?][10, 0, 0, 100, 12, 2]
# [Ongoing][10, 0, 0, 100, 16, 2]
# [Ongoing][10, 0, 0, 100, 32 2]

# [?][10, 0, 0, 10, 16, 2]: 77.03%, overlap 1 / decomp lowest 81.6%
# [X?][10, 0, 0, 10, 32, 2]_cont: 84.15%, overlap 0 / decomp lowest 90% 

# [X][10, 0, 0, 100, 16, 2]: 79.25%, overlap 2 / decomp lowest 93%
# [X][10, 0, 0, 10, 64, 2]_cont: 61.03%, overlap 0  / decomp lowest 97.08% 
# [X]2nd[10, 0, 0, 100, 64, 2]: 67.9%, overlap 0 / decomp lowest 95% 
# [X][10, 0, 0, 1000, 16, 2] 
# [X][10, 0, 0, 1000, 32, 2] 
# [X][100, 0, 0, 1, 16, 2]: 30%
# [X][100, 0, 0, 1, 32, 2]: 15%

# w sub
# [10, 1, 0, 1, 16, 10]: Nan
# [10, 1, 0, 1, 16, 5]: 85.18%, overlap 2 / decomp lowest 81%
# [10, 1, 0, 10, 16, 10]: 79.85%, overlap 2  / decomp lowest 77.3%
# [10, 1, 0, 10, 16, 5]: nan
# ~ nan
# [10, 10, 0, 10, 16, 10]: 82.44%, overlap 4  / decomp lowest 70.6%
# [10, 10, 0, 1, 32, 10]: 36.90%, overlap 2 / decomp lowest 54.7%
# [10, 10, 0, 1, 32, 5]: 30.99%, overlap many / decomp lowest 1.32%
# [10, 10, 0, 10, 32, 10]: nan


Test KAE

In [ ]:
loss_kae, z = compute_l_kae(kae, params_snapshots) # Koopman operator is updated here.
N_O = z.shape[-1]
param_sub_all, eigvals = compute_theta_sub_all(kae, z, kae.K)

test_classifier(classifier, test_loader)
bests = np.zeros(num_classes)
best_index = np.zeros(num_classes)
best_count = np.zeros(num_classes)

print('------------------------')
with torch.autograd.no_grad():
    param_sub_all, eigvals = compute_theta_sub_all(kae, z, kae.K)
    eigvals_d = eigvals.cpu().detach()

    for i in range(0,hidden_k):
        if i == 0:
            param_sub_kae = param_sub_all[:, 0]
        else:
            param_sub_kae = param_sub_kae + param_sub_all[:, i]
    kae_classifier = MLP(image_size, hidden_c, num_classes).to(device)
    kae_classifier.eval()
    nn.utils.vector_to_parameters(param_sub_kae, kae_classifier.parameters())
    print('KAE accuracy ----')
    test_classifier(kae_classifier, test_loader)

    for idx_modes in range(hidden_k):
        temp = np.zeros(num_classes)

        for idx_sub in range(num_classes):
            param_sub = param_sub_all[:, idx_modes]
            classifier_sub_test = MLP(image_size, hidden_c, num_classes).to(device)
            classifier_sub_test.eval()
            nn.utils.vector_to_parameters(param_sub, classifier_sub_test.parameters())
            testloader = dataset_in_use_per_class.sub_testloaders[idx_sub]
            total = 0
            correct = 0
            for images, labels in testloader:
                images = images.reshape(-1, 28*28).to(device)
                labels = labels.to(device)
                outputs = classifier_sub_test(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
            accuracy = 100 * correct / total
            temp[idx_sub] = accuracy
        formatted = np.array([num for num in temp])

        if idx_modes == 0:
            formatted_history = formatted
        else:
            formatted_history = np.vstack((formatted_history, formatted))

# Get best element
formatted_history2 = np.array(formatted_history, dtype=float)
bests = np.max(formatted_history2,axis = 0)
is_max = formatted_history2 == bests
row_counts = np.sum(is_max, axis = 1)

print('Best elements :')
print(bests)
print('Best counts :')
print(row_counts)

num_cols = formatted_history2.shape[1]
for col in range(num_cols):
    rows_with_max = np.where(formatted_history2[:,col] == bests[col])[0]
    print('------------')
    print(f'Class {col}: Best = {bests[col]}')
    for row_idx in rows_with_max:
        print(f'Mode {row_idx}: [|eig|={torch.linalg.norm(eigvals_d[row_idx])}] {np.array2string(formatted_history2[row_idx], precision=2)}')

Eigenvalue plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
circle = plt.Circle((0,0),1,fill=False)
ax.add_artist(circle)
ax.set_xlim(-1.5,1.5)
ax.set_ylim(-1.5,1.5)
ax.scatter(eigvals_d.real, eigvals_d.imag)

eig_count = 0
for i in range(0,hidden_k):
    if torch.linalg.norm(eigvals_d[i])<0.1:
        eig_count +=1

print(str(eig_count)+'-minor eigvals')

Partial reconstruction

In [ ]:
# Manually generate the partially reconstructed parameter here
param_sub = param_sub_all[:, 29] + param_sub_all[:, 32] + param_sub_all[:, 37] + param_sub_all[:, 72]


bests = np.zeros(num_classes)
best_index = np.zeros(num_classes)
best_count = np.zeros(num_classes)

print('------------------------')
with torch.autograd.no_grad():
    param_sub_all, eigvals = compute_theta_sub_all(kae, z, kae.K)
    eigvals_d = eigvals.cpu().detach()
    eigvals_d = eigvals_d[0:hidden_k]
    order = torch.argsort(eigvals.abs())
    candidates = torch.linspace(0, 9, 10, dtype=int, device=device)
    temp = np.zeros(num_classes)

    classifier_sub_test = MLP(image_size, hidden_c, num_classes).to(device)
    classifier_sub_test.eval()
    nn.utils.vector_to_parameters(param_sub, classifier_sub_test.parameters())

    for idx_sub in range(num_classes):        

        testloader = mnist_per_class.sub_testloaders[idx_sub]
        total = 0
        correct = 0
        for images, labels in testloader:
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)
            outputs = classifier_sub_test(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        accuracy = 100 * correct / total
        temp[idx_sub] = accuracy
    # formatted = np.array([f"{num:.6f}" for num in temp])
    formatted = np.array([num for num in temp])

print(formatted)